# FarmLens Plant Disease Detection - Model Demo

This notebook demonstrates the training and testing process of the MobileNetV2 model used in FarmLens.

**Author**: FarmLens Team  
**Date**: 2026  
**Model**: MobileNetV2 (Transfer Learning)  
**Dataset**: PlantVillage (38 classes)

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
import json
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Load Pre-trained Model

In [ ]:
# Load class names
with open('class_names.json', 'r') as f:
    class_names = json.load(f)

print(f"Number of classes: {len(class_names)}")
print(f"\nFirst 10 classes:")
for i, name in enumerate(class_names[:10], 1):
    print(f"  {i}. {name}")

In [ ]:
# Build model architecture
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.classifier[1].in_features, len(class_names))
)

# Load trained weights
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.load_state_dict(torch.load('mobilenetv2_plant.pth', map_location=device))
model = model.to(device)
model.eval()

print("✓ Model loaded successfully!")
print(f"✓ Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Single Image Prediction Demo

In [ ]:
def predict_image(image_path, model, class_names, device):
    """
    Predict disease from a single image
    """
    # Preprocessing
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # Load and transform image
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1)[0]
        
    # Get top 5 predictions
    top5_probs, top5_indices = torch.topk(probs, 5)
    
    # Display results
    plt.figure(figsize=(12, 4))
    
    # Show image
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f"Input Image\n{image_path}", fontsize=12)
    plt.axis('off')
    
    # Show predictions
    plt.subplot(1, 2, 2)
    top5_names = [class_names[idx] for idx in top5_indices]
    top5_probs_np = top5_probs.cpu().numpy()
    
    colors = ['green' if i == 0 else 'skyblue' for i in range(5)]
    plt.barh(range(5), top5_probs_np, color=colors)
    plt.yticks(range(5), [f"{name[:30]}..." if len(name) > 30 else name for name in top5_names])
    plt.xlabel('Confidence')
    plt.title('Top 5 Predictions', fontsize=12)
    plt.xlim(0, 1)
    
    for i, prob in enumerate(top5_probs_np):
        plt.text(prob + 0.02, i, f'{prob:.2%}', va='center')
    
    plt.tight_layout()
    plt.show()
    
    # Print results
    print(f"\n{'='*70}")
    print(f"PREDICTION RESULTS")
    print(f"{'='*70}")
    print(f"\nTop Prediction: {top5_names[0]}")
    print(f"Confidence: {top5_probs_np[0]:.2%}")
    print(f"\nTop 5 Predictions:")
    for i, (name, prob) in enumerate(zip(top5_names, top5_probs_np), 1):
        print(f"  {i}. {name:45s} {prob:.2%}")
    print(f"{'='*70}\n")

# Example usage (replace with your image path)
# predict_image('path/to/your/plant/image.jpg', model, class_names, device)

## 4. Batch Testing on Test Set

In [ ]:
# Prepare test data loader
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load test dataset (adjust path as needed)
test_dataset = datasets.ImageFolder(
    root='./dataset/test',  # or './dataset/valid'
    transform=test_transforms
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: 32")
print(f"Number of batches: {len(test_loader)}")

In [ ]:
# Run inference on test set
all_preds = []
all_labels = []

print("Running inference on test set...")
with torch.no_grad():
    for batch_idx, (inputs, labels) in enumerate(test_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  Processed {(batch_idx + 1) * 32} samples...")

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print("\n✓ Inference complete!")

## 5. Evaluation Metrics

In [ ]:
# Calculate accuracy
accuracy = np.mean(all_preds == all_labels)
print(f"Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Classification report
print("\nDetailed Classification Report:")
print("=" * 70)
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

## 6. Confusion Matrix Visualization

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot
plt.figure(figsize=(20, 18))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Accuracy'}
)
plt.title('Confusion Matrix (Normalized)', fontsize=16, pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Per-Class Performance Analysis

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Calculate per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(
    all_labels, all_preds, average=None, zero_division=0
)

# Create DataFrame for better visualization
import pandas as pd

metrics_df = pd.DataFrame({
    'Class': class_names,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support.astype(int)
})

# Sort by F1-Score
metrics_df = metrics_df.sort_values('F1-Score', ascending=False)

print("\nTop 10 Best Performing Classes:")
print(metrics_df.head(10).to_string(index=False))

print("\nTop 10 Worst Performing Classes:")
print(metrics_df.tail(10).to_string(index=False))

## 8. Model Performance Summary

In [ ]:
print("="*70)
print("MODEL PERFORMANCE SUMMARY")
print("="*70)
print(f"\nModel: MobileNetV2 (Transfer Learning)")
print(f"Dataset: PlantVillage")
print(f"Number of Classes: {len(class_names)}")
print(f"Test Samples: {len(all_labels)}")
print(f"\nOverall Metrics:")
print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Precision: {precision.mean():.4f}")
print(f"  Recall:    {recall.mean():.4f}")
print(f"  F1-Score:  {f1.mean():.4f}")
print(f"\nModel Size: 9.3 MB")
print(f"Parameters: ~3.5 million")
print(f"Inference Time: ~50-100ms per image (CPU)")
print("="*70)

## 9. Conclusion

This notebook demonstrated:
1. Loading the pre-trained MobileNetV2 model
2. Making predictions on single images
3. Batch testing on the test set
4. Evaluating model performance with various metrics
5. Visualizing results with confusion matrix

The model achieves **85-95% accuracy** on the PlantVillage dataset and is integrated into the FarmLens application as a reliable fallback when the Gemini API is unavailable.

### Next Steps:
- Fine-tune the model with additional disease classes
- Experiment with different architectures (ResNet, EfficientNet)
- Implement model quantization for faster inference
- Deploy to mobile devices using TorchScript or ONNX